# Compress ResNet-56 (CIFAR-10) — complete Kaggle notebook

`ResNet-56 (FP32, 56 layers)  →  gradual prune + fine-tune  →  INT8 quantize  →  ONNX  →  verify against the original`

The ResNet-56 baseline (92%+ on CIFAR-10) ships inside the repo, so **no training step**.

**Before Run All** (right panel): *Accelerator → GPU T4*, *Internet → On*.
Optional but saves ~30 min: *Add Input → search "CIFAR-10"* and attach any CIFAR-10 dataset in the
original python-pickle format (it must contain a `cifar-10-batches-py` folder). Otherwise it downloads (170 MB).

Everything lands in `/kaggle/working/` — download `resnet56_compressed.zip` from the *Output* tab.

In [1]:
# ======================= CONFIG =======================
PRUNE_RATIO = 0.5    # fraction of channels removed (0.5 channels ~ 75% fewer parameters)
STEPS       = 5      # gradual pruning steps
FT_EPOCHS   = 5      # fine-tune epochs after every step (25 total)
LOSS        = "dkd"  # "kd" | "dkd"
# ======================================================

In [2]:
import os, subprocess, sys, json, shutil, pathlib
REPO = "https://github.com/Bhagyesh0603/Model-Compression-ToolKit..git"
if not os.path.exists("/kaggle/working/ck"):
    subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/ck"], check=True)
os.chdir("/kaggle/working/ck")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> enable GPU in the right panel, then Restart & Run All")
from compresskit.zoo.data import resolve_root
print("CIFAR-10 location:", resolve_root(), "(attached dataset if under /kaggle/input, else it will download)")
assert os.path.exists("checkpoints/resnet56_fp32.pt"), "baseline checkpoint missing from the repo"

def run(*args):
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    return subprocess.run(args).returncode

GPU: NONE -> enable GPU in the right panel, then Restart & Run All
CIFAR-10 location: /kaggle/working/ck/data (attached dataset if under /kaggle/input, else it will download)


## 1 · Compress: gradual prune + fine-tune → INT8 → evaluate the .onnx

In [3]:
OUT = "/kaggle/working"
code_ = run(sys.executable, "-m", "compresskit.cli", "pipeline",
            "--model", "checkpoints/resnet56_fp32.pt",
            "--prune", PRUNE_RATIO, "--steps", STEPS, "--fine-tune-epochs", FT_EPOCHS, "--loss", LOSS,
            "--precision", "int8", "--allow-slow-int8",
            "--workdir", f"{OUT}/work", "--report", f"{OUT}/report.html", "--json", f"{OUT}/results.json")
assert code_ == 0, "pipeline failed - read the log above"

$ /usr/bin/python3 -m compresskit.cli pipeline --model checkpoints/resnet56_fp32.pt --prune 0.5 --steps 5 --fine-tune-epochs 5 --loss dkd --precision int8 --allow-slow-int8 --workdir /kaggle/working/work --report /kaggle/working/report.html --json /kaggle/working/results.json
loaded checkpoints/resnet56_fp32.pt (resnet56, baseline accuracy 93.18%)


100%|██████████| 170M/170M [51:02<00:00, 55.7kB/s]


PRE-FLIGHT
host      : Intel(R) Xeon(R) CPU @ 2.20GHz (x86_64, Linux)
cores     : 4 logical / 4 performance
ISA       : avx2, f16c
INT8      : slow — No VNNI/AMX/DotProd/I8MM on this CPU — INT8 may be SLOWER than FP32. Benchmark before shipping it.
default   : fp16
versions  : python 3.13.15, torch 2.11.0+cpu, onnxruntime 1.31.1, numpy 2.1.3

model     : 855,770 parameters, input (1, 3, 32, 32)
calibration: 512 samples (must come from the train split)
protected : classifier ['Linear'] is never pruned
plan      : prune -> recover -> quantize

baseline               fp32  acc  93.18%  lat   4.655 ms (p95   5.183)  size   3.28 MB  params 855,770
  [step 1/5] fine-tuning 5 epoch(s) (dkd)...
    [step 1] epoch 1/5 | loss: 5.3247 | train_acc: 93.99% | lr: 0.00499
    [step 1] epoch 2/5 | loss: 6.4650 | train_acc: 90.94% | lr: 0.00427
    [step 1] epoch 3/5 | loss: 4.2731 | train_acc: 94.28% | lr: 0.00251
    [step 1] epoch 4/5 | loss: 2.8326 | train_acc: 96.36% | lr: 0.00074
    [step 1] epo

## 2 · Does the compressed model actually work? (original vs ONNX on 2000 real test images)

In [4]:
r = json.load(open(f"{OUT}/results.json"))
final_onnx = r["stages"][-1]["artifact"]
shutil.copy(final_onnx, f"{OUT}/model.onnx")
shutil.copy("checkpoints/resnet56_fp32.pt", f"{OUT}/resnet56_fp32.pt")
run(sys.executable, "experiments/verify_onnx.py", "--model", "checkpoints/resnet56_fp32.pt",
    "--onnx", f"{OUT}/model.onnx", "--images", 2000)

$ /usr/bin/python3 experiments/verify_onnx.py --model checkpoints/resnet56_fp32.pt --onnx /kaggle/working/model.onnx --images 2000

Tested on 2000 real test images (cifar10)

                        original (PyTorch)   compressed (ONNX)
accuracy                            93.05%              89.90%
agrees with original                                    91.65%
file size                          3.58 MB             0.33 MB
latency (this CPU)                                    2.022 ms

Side by side (✓ = correct):
  #  true          original        compressed      
  0  cat           cat ✓           cat ✓           
  1  ship          ship ✓          ship ✓          
  2  ship          ship ✓          ship ✓          
  3  airplane      airplane ✓      airplane ✓      
  4  frog          frog ✓          frog ✓          
  5  frog          frog ✓          frog ✓          
  6  automobile    automobile ✓    automobile ✓    
  7  frog          frog ✓          frog ✓          

PASS: compre

0

## 3 · Summary and download bundle

In [5]:
b, f, t = r["baseline"], r["stages"][-1], r["totals"]
print(f"accuracy : {b['accuracy']*100:.2f}% -> {f['accuracy']*100:.2f}%   (drop {t['accuracy_drop_pp']:.2f} pp)")
print(f"size     : {b['size_mb']:.2f} MB -> {f['size_mb']:.2f} MB   ({t['size_reduction']:.1f}x smaller)")
print(f"precision: {f['precision']}")
print("NOTE: latency in this run is Kaggle-CPU latency, not your device. Measure on the target: compresskit bench model.onnx")

import zipfile
with zipfile.ZipFile(f"{OUT}/resnet56_compressed.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("model.onnx", "report.html", "results.json", "resnet56_fp32.pt"):
        z.write(f"{OUT}/{name}", name)
print("bundle:", f"{OUT}/resnet56_compressed.zip", f"({os.path.getsize(f'{OUT}/resnet56_compressed.zip')/1e6:.1f} MB)")
from IPython.display import IFrame, display
display(IFrame(f"{OUT}/report.html", width="100%", height=900))

accuracy : 93.18% -> 89.28%   (drop 3.90 pp)
size     : 3.28 MB -> 0.33 MB   (9.8x smaller)
precision: int8
NOTE: latency in this run is Kaggle-CPU latency, not your device. Measure on the target: compresskit bench model.onnx
bundle: /kaggle/working/resnet56_compressed.zip (3.7 MB)


## 4 · On your Mac (real speed check, once)

```bash
unzip resnet56_compressed.zip
python experiments/verify_onnx.py --model resnet56_fp32.pt --onnx model.onnx
compresskit bench model.onnx
```